In [3]:
import sys
from pathlib import Path

# Add project root to Python path
ROOT_DIR = Path.cwd().resolve().parent
sys.path.insert(0, str(ROOT_DIR))


import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from concurrent.futures import ThreadPoolExecutor
from src.evaluation import get_llm_response, get_llm_response_retry, calculate_price,calculate_total_price, map_progress
from src.ingest import load_machine_data

GROUND_TRUTH_OUTPUT_PATH = ROOT_DIR / "data"
RAG_ANSWER_PATH = ROOT_DIR / "data"
RAG_EVALUATE_PATH = ROOT_DIR / "data"

%load_ext autoreload
%autoreload 2

In [4]:
PROJECT_ROOT = Path.cwd().parent
GROUND_TRUTH_OUTPUT_PATH = ROOT_DIR / "data"
GROUND_TRUTH_OUTPUT_PATH 

WindowsPath('D:/DataTalksClub/Submission/Predictive-Maintenance-RAG-Assistant/data')

# 1. Generating ground_truth file

In [5]:
# load machine data
documents = load_machine_data()

Loading knowledge base from json file: D:\DataTalksClub\Submission\Predictive-Maintenance-RAG-Assistant\data\knowledge_base.json
Loaded machine data:71


In [6]:

# Set to a specific section name (e.g. 'Power Failure') to only generate
# ground-truth questions for that section; set to None (default) to generate
# over the whole knowledge base instead of just one failure mode.
SECTION_FILTER = None
# SECTION_FILTER = 'Power Failure'

documents_to_process = [
    doc for doc in documents
    if SECTION_FILTER is None or doc['section'] == SECTION_FILTER
]
len(documents_to_process)

doc = documents_to_process[0]
print(f'id : {doc['id']}')
print(f"section : {doc['section']}")
print(f"question: {doc['question']}")
print(f"answer: {doc['answer']}")

user_prompt= json.dumps(doc)
user_prompt

id : 299b2d2994
section : Tool Wear Failure
question: What are the symptoms of Tool Wear Failure?
answer: Tool wear time accumulates into a critical band, typically between 200 and 240 minutes of continuous use. Within this band, failure probability rises sharply and can occur without any other sensor anomaly — torque, temperature, and rotational speed may all look nominal right up to the failure event.


'{"id": "299b2d2994", "process_control": "predictive-maintenance", "failure_mode": "TWF", "section": "Tool Wear Failure", "question": "What are the symptoms of Tool Wear Failure?", "answer": "Tool wear time accumulates into a critical band, typically between 200 and 240 minutes of continuous use. Within this band, failure probability rises sharply and can occur without any other sensor anomaly \\u2014 torque, temperature, and rotational speed may all look nominal right up to the failure event."}'

In [5]:
# get OpenAI response
openai_client = OpenAI()

class Questions(BaseModel):
    questions : list[str]

model="gpt-5.4-mini"

# build data generation instructions to OpenAI
data_gen_instructions = """
You emulate a machine operator/ equipment enginner who's taking care of machine.
Formulate 5 questions this machine operator/ equipment enginner might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

messages = [
    {'role':'developer', 'content': data_gen_instructions},
    {'role':'user','content':user_prompt}
]

response = openai_client.responses.parse(
    model = model,
    input=messages,
    text_format= Questions
)

response.output_parsed.questions

['What are the usual signs that a power failure is happening in the process?',
 'At what power level does the system get flagged for being underpowered or stalling?',
 'When does the process count as overpowered and at risk of mechanical overload?',
 'How can I tell if the delivered power is outside the safe operating range?',
 'What power limits should I watch to avoid incomplete cutting or overload issues?']

In [6]:
# simulate just like user ask questions from frontend application
result, usage = get_llm_response(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions,
    model
)

# calculate price
calculate_price(usage)


{'input_cost': 0.000228, 'output_cost': 0.000522, 'total_cost': 0.00075}

In [7]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)
    results=[]

    llm_output, llm_usage = get_llm_response_retry(openai_client, data_gen_instructions,user_prompt,Questions, model)

    for q in llm_output.questions:
        results.append({
            'question': q,
            'query_section':doc['section'],
            'document':doc['id']})

    return results, llm_usage

In [8]:
# prepare for all possible questions(=documents) related to section 'Power Failure'
with ThreadPoolExecutor(max_workers=6) as pool:
    results=map_progress(pool, documents,generate_ground_truth)

  0%|          | 0/71 [00:00<?, ?it/s]

In [9]:
# prepare ground_truth
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

355

In [10]:
# calculate total price
total_cost = calculate_total_price(usages)
total_cost

0.0552945

In [11]:
# save ground_truth data file
df_ground_truth = pd.DataFrame(ground_truth)

df_ground_truth.to_csv(GROUND_TRUTH_OUTPUT_PATH / "ground_truth.csv", index=False)